# Docker e Kubernetes
## Do zero ao container em produção

---

Este notebook explica o que são **Docker** e **Kubernetes**, por que existem, e os **principais comandos** de cada um — com exemplos diretos do projeto de fraude.

---
## 1. O Problema que o Docker Resolve

Você já passou por isso:

```
Desenvolvedor A: "Funciona na minha máquina!"
Desenvolvedor B: "Pois no servidor dá erro."
Servidor prod:   ModuleNotFoundError: No module named 'xgboost'
```

O problema é simples: **diferentes máquinas têm ambientes diferentes**.

| Componente | Dev A | Dev B | Servidor |
|---|---|---|---|
| Python | 3.11 | 3.9 | 3.8 |
| XGBoost | 2.1.1 | 2.0.0 | não instalado |
| OS | macOS | Ubuntu 22 | CentOS 7 |

**Docker** resolve isso empacotando **o código + todas as dependências + o sistema operacional** em uma unidade portátil chamada **container**. Se funciona no seu PC, funciona no servidor — garantido.

---
## 2. Conceitos Fundamentais do Docker

### 2.1 Imagem (Image)

Uma **imagem Docker** é um template somente-leitura que define o ambiente. É construída a partir de um `Dockerfile` e contém:
- Sistema operacional base (ex: Debian, Alpine)
- Runtime (ex: Python 3.11)
- Dependências instaladas (ex: xgboost, fastapi)
- Código da aplicação

Pense na imagem como uma **receita de bolo**: ela descreve como criar o ambiente, mas não é o ambiente em si.

### 2.2 Container

Um **container** é uma imagem em execução. Você pode criar múltiplos containers a partir da mesma imagem, e cada um roda de forma isolada.

Pense no container como o **bolo assado**: feito a partir da receita (imagem), mas agora é uma instância real e em execução.

```
Dockerfile  →  docker build  →  Imagem  →  docker run  →  Container
(receita)                      (template)                 (instância)
```

### 2.3 Registry

Um **registry** é um repositório de imagens Docker, assim como o GitHub é um repositório de código. O mais comum é o **Docker Hub** (hub.docker.com).

```
docker push → envia imagem para o registry
docker pull → baixa imagem do registry
```

### 2.4 Volume

Containers são **efêmeros**: quando o container é destruído, todos os dados dentro dele somem. Para persistir dados, usamos **volumes** — diretórios do host montados dentro do container.

### 2.5 Camadas (Layers)

Imagens Docker são compostas de **camadas imutáveis**. Cada instrução no Dockerfile cria uma nova camada. O Docker usa cache: se uma camada não mudou, ela é reutilizada na próxima build — tornando o processo muito mais rápido.

```dockerfile
FROM python:3.11-slim      ← camada 1 (base)
RUN pip install mlflow     ← camada 2 (dependências — cache reaproveitado se não mudar)
COPY src/ src/             ← camada 3 (código — muda frequentemente)
```

---
## 3. Anatomia de um Dockerfile

O `Dockerfile.train` do projeto, linha a linha:

```dockerfile
# Imagem base: Python 3.11 com Debian 12 (Bookworm)
# "slim" = versão mínima, sem ferramentas extras (~50MB vs ~300MB do full)
FROM python:3.11-slim-bookworm

# Define o diretório de trabalho dentro do container
# Todos os comandos subsequentes executam a partir daqui
WORKDIR /app

# Copia o arquivo de requirements ANTES do código
# Por quê? Se o requirements.txt não mudou, o Docker reutiliza
# o cache da camada de pip install — builds muito mais rápidos
COPY requirements-train.txt .

# Instala as dependências
# --no-cache-dir: não salva cache do pip dentro da imagem (mantém menor)
RUN pip install --no-cache-dir -r requirements-train.txt

# Copia o código fonte
# Feito por último: mudanças no código não invalidam o cache de dependências
COPY src/ src/

# Comando executado quando o container sobe
CMD ["python", "src/train.py"]
```

### Instruções mais comuns do Dockerfile

| Instrução | O que faz | Exemplo |
|---|---|---|
| `FROM` | Define a imagem base | `FROM python:3.11-slim` |
| `WORKDIR` | Define o diretório de trabalho | `WORKDIR /app` |
| `COPY` | Copia arquivos do host para a imagem | `COPY src/ src/` |
| `RUN` | Executa um comando durante o build | `RUN pip install mlflow` |
| `ENV` | Define variável de ambiente | `ENV PORT=8000` |
| `EXPOSE` | Documenta a porta que o container usa | `EXPOSE 8000` |
| `CMD` | Comando padrão ao subir o container | `CMD ["python", "app.py"]` |
| `ENTRYPOINT` | Similar ao CMD, mas não pode ser sobrescrito | `ENTRYPOINT ["uvicorn"]` |

---
## 4. Comandos Essenciais do Docker

### 4.1 Build e Push de Imagens

```bash
# Construir uma imagem a partir do Dockerfile
# -f: especifica qual Dockerfile usar
# -t: nomeia a imagem (usuario/nome:tag)
# . : contexto de build (diretório atual)
docker build -f docker/Dockerfile.train -t thiagoar587/fraud-train:latest .

# Enviar imagem para o Docker Hub
docker push thiagoar587/fraud-train:latest

# Baixar imagem do registry
docker pull python:3.11-slim

# Listar imagens locais
docker images

# Remover uma imagem
docker rmi thiagoar587/fraud-train:latest

# Remover imagens, containers e cache não utilizados
docker system prune -a
```

### 4.2 Executar Containers

```bash
# Rodar um container
docker run thiagoar587/fraud-train:latest

# Flags mais usadas:
# -d: modo detached (roda em background)
# -p: mapeamento de porta (host:container)
# -e: variável de ambiente
# -v: monta um volume (host_path:container_path)
# --name: nomeia o container
# --rm: remove o container automaticamente ao finalizar

# Exemplo: subir a API localmente
docker run -d \
  --name fraud-api \
  -p 8000:8000 \
  -e MLFLOW_TRACKING_URI=http://localhost:5000 \
  thiagoar587/fraud-serve:latest

# Rodar container interativamente (abre um shell)
docker run -it thiagoar587/fraud-train:latest bash

# Rodar o treinamento com volume montado
docker run --rm \
  -v $(pwd)/data:/data \
  -e DATA_PATH=/data/transactions_1M.csv \
  thiagoar587/fraud-train:latest
```

### 4.3 Gerenciar Containers em Execução

```bash
# Listar containers rodando
docker ps

# Listar todos (incluindo parados)
docker ps -a

# Ver logs de um container
docker logs fraud-api

# Seguir logs em tempo real
docker logs -f fraud-api

# Executar comando dentro de um container rodando
docker exec -it fraud-api bash

# Parar um container
docker stop fraud-api

# Remover um container parado
docker rm fraud-api

# Ver uso de recursos (CPU, memória) dos containers
docker stats

# Inspecionar detalhes de um container
docker inspect fraud-api
```

### 4.4 Volumes e Redes

```bash
# Criar um volume nomeado
docker volume create mlflow-data

# Listar volumes
docker volume ls

# Usar volume nomeado no container
docker run -v mlflow-data:/mlflow thiagoar587/mlflow-server:latest

# Listar redes
docker network ls

# Criar rede (permite containers se comunicarem por nome)
docker network create mlops-net

# Conectar dois containers na mesma rede
docker run -d --network mlops-net --name mlflow thiagoar587/mlflow-server:latest
docker run -d --network mlops-net -e MLFLOW_TRACKING_URI=http://mlflow:5000 thiagoar587/fraud-serve:latest
```

---
## 5. O que é Kubernetes?

Docker resolve o empacotamento. Mas e quando você tem:
- **100 containers** para gerenciar?
- Um container que **cai** e precisa reiniciar automaticamente?
- Necessidade de **escalar** de 1 para 10 instâncias durante pico de tráfego?
- Containers que precisam se **descobrir** e se comunicar?

**Kubernetes (K8s)** é um orquestrador de containers: um sistema que gerencia automaticamente o ciclo de vida de containers em um cluster de máquinas.

### Analogia: Docker vs Kubernetes

| Analogia | Docker | Kubernetes |
|---|---|---|
| Marítima | Container de carga | Porto que organiza os containers |
| Restaurante | Receita e prato pronto | Gerente que escala cozinheiros e garçons |
| TI | Um processo rodando | Sistema operacional distribuído |

### O que Kubernetes faz automaticamente

| Problema | Solução K8s |
|---|---|
| Container caiu | Reinicia automaticamente |
| Tráfego aumentou | Escala o número de réplicas |
| Novo deploy quebrou | Rollback automático para versão anterior |
| Onde está o IP do serviço X? | DNS interno: acesse pelo nome `mlflow-svc` |
| Como distribuir tráfego entre réplicas? | Load balancing automático via Services |
| Como persistir dados além do container? | PersistentVolumes |
| Quanto de CPU/RAM cada container pode usar? | Resources limits/requests |

---
## 6. Arquitetura do Kubernetes

```
┌────────────────────────────────────────────────────────────────┐
│                    KUBERNETES CLUSTER                          │
│                                                                │
│  ┌─────────────────────────────┐                               │
│  │       CONTROL PLANE         │   ← "O Cérebro"               │
│  │                             │                               │
│  │  API Server   ← kubectl     │   ← recebe seus comandos      │
│  │  Scheduler                  │   ← decide em qual nó rodar   │
│  │  etcd                       │   ← banco de dados do cluster │
│  │  Controller Manager         │   ← garante o estado desejado │
│  └─────────────────────────────┘                               │
│                                                                │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐         │
│  │   NODE 1     │  │   NODE 2     │  │   NODE 3     │         │
│  │  ┌────────┐  │  │  ┌────────┐  │  │  ┌────────┐  │         │
│  │  │  Pod   │  │  │  │  Pod   │  │  │  │  Pod   │  │         │
│  │  │(train) │  │  │  │(serve) │  │  │  │(mlflow)│  │         │
│  │  └────────┘  │  │  └────────┘  │  │  └────────┘  │         │
│  │  kubelet     │  │  kubelet     │  │  kubelet     │         │
│  └──────────────┘  └──────────────┘  └──────────────┘         │
└────────────────────────────────────────────────────────────────┘
```

### Recursos principais do K8s (em ordem de abstração)

| Recurso | O que é | Usado para |
|---|---|---|
| **Namespace** | Ambiente isolado dentro do cluster | Separar projetos |
| **Pod** | Menor unidade: um ou mais containers | Rodar a aplicação |
| **Deployment** | Gerencia Pods, garante N réplicas sempre rodando | Serviços contínuos (APIs) |
| **Job** | Roda um Pod até ele completar com sucesso | Tarefas batch (treinamento) |
| **Service** | Endpoint estável (IP + DNS) para acessar Pods | Comunicação entre serviços |
| **PersistentVolumeClaim** | "Pedido" de disco persistente | Banco de dados, arquivos |
| **ConfigMap** | Configurações não-secretas | URLs, parâmetros |
| **Secret** | Configurações sensíveis | Senhas, tokens, chaves API |

---
## 7. Comandos Essenciais do kubectl

O `kubectl` é a CLI para interagir com o cluster Kubernetes. A sintaxe geral é:

```
kubectl [ação] [tipo-de-recurso] [nome] [flags]
```

### 7.1 Contexto e Cluster

```bash
# Ver qual cluster está sendo usado
kubectl config current-context

# Listar todos os contextos disponíveis
kubectl config get-contexts

# Trocar de contexto
kubectl config use-context minikube

# Informações gerais do cluster
kubectl cluster-info

# Listar nós do cluster
kubectl get nodes
```

### 7.2 Namespaces

```bash
# Listar namespaces
kubectl get namespaces

# Criar namespace
kubectl create namespace mlops-demo

# Usar namespace em todos os comandos (-n = --namespace)
kubectl get pods -n mlops-demo

# Definir namespace padrão para a sessão
kubectl config set-context --current --namespace=mlops-demo

# Remover namespace (e TUDO dentro dele!)
kubectl delete namespace mlops-demo
```

### 7.3 Apply e Delete (Manifests)

```bash
# Aplicar um manifest (cria ou atualiza o recurso)
kubectl apply -f k8s/00-namespace.yaml

# Aplicar todos os manifests de um diretório
kubectl apply -f k8s/

# Remover recurso pelo manifest
kubectl delete -f k8s/04-train-job.yaml

# Remover recurso pelo nome
kubectl delete pod meu-pod -n mlops-demo
kubectl delete deployment fraud-serving -n mlops-demo
```

### 7.4 Inspecionar Recursos

```bash
# Listar pods
kubectl get pods -n mlops-demo

# Mais detalhes (IP, nó, tempo rodando)
kubectl get pods -n mlops-demo -o wide

# Watch: atualiza automaticamente a cada 2 segundos
kubectl get pods -n mlops-demo -w

# Inspecionar um pod (detalhes completos: eventos, volumes, env vars)
kubectl describe pod fraud-train-job-xxxxx -n mlops-demo

# Listar outros recursos
kubectl get deployments -n mlops-demo
kubectl get services -n mlops-demo
kubectl get pvc -n mlops-demo
kubectl get jobs -n mlops-demo

# Ver todos os recursos do namespace
kubectl get all -n mlops-demo
```

### 7.5 Logs e Debug

```bash
# Ver logs de um pod
kubectl logs fraud-train-job-xxxxx -n mlops-demo

# Seguir logs em tempo real (como docker logs -f)
kubectl logs -f fraud-train-job-xxxxx -n mlops-demo

# Logs de um Job (sem precisar do nome exato do pod)
kubectl logs -f job/fraud-train-job -n mlops-demo

# Logs de um Deployment (de qualquer pod da réplica)
kubectl logs -f deployment/fraud-serving -n mlops-demo

# Últimas 100 linhas dos logs
kubectl logs --tail=100 fraud-train-job-xxxxx -n mlops-demo

# Abrir shell interativo dentro de um pod rodando
kubectl exec -it fraud-serving-xxxxx -n mlops-demo -- bash

# Executar um comando sem abrir shell
kubectl exec data-uploader -n mlops-demo -- ls -lh /data
```

### 7.6 Port-Forward (Acessar Serviços Localmente)

```bash
# Redirecionar porta do serviço K8s para o localhost
# Útil para desenvolvimento e debug

# Acessar o MLflow UI em http://localhost:5000
kubectl port-forward svc/mlflow-svc 5000:5000 -n mlops-demo

# Acessar a API de serving em http://localhost:8000
kubectl port-forward svc/fraud-serving-svc 8000:8000 -n mlops-demo

# Port-forward de um pod diretamente
kubectl port-forward pod/mlflow-xxxxx 5000:5000 -n mlops-demo
```

### 7.7 Escalar e Atualizar Deployments

```bash
# Escalar para 3 réplicas
kubectl scale deployment fraud-serving --replicas=3 -n mlops-demo

# Atualizar a imagem de um deployment (novo deploy rolling update)
kubectl set image deployment/fraud-serving api=thiagoar587/fraud-serve:v2 -n mlops-demo

# Acompanhar o rollout
kubectl rollout status deployment/fraud-serving -n mlops-demo

# Fazer rollback para a versão anterior
kubectl rollout undo deployment/fraud-serving -n mlops-demo

# Histórico de deploys
kubectl rollout history deployment/fraud-serving -n mlops-demo
```

### 7.8 Copiar Arquivos

```bash
# Copiar arquivo do host para dentro do pod
kubectl cp data/transactions_1M.csv data-uploader:/data/transactions_1M.csv -n mlops-demo

# Copiar arquivo de dentro do pod para o host
kubectl cp mlops-demo/mlflow-xxxxx:/mlflow/mlflow.db ./mlflow_backup.db
```

### 7.9 Aguardar Condições

```bash
# Aguardar até pod ficar Ready
kubectl wait --for=condition=Ready pod/data-uploader -n mlops-demo --timeout=60s

# Aguardar Job completar
kubectl wait --for=condition=complete job/fraud-train-job -n mlops-demo --timeout=600s

# Aguardar Deployment estar disponível
kubectl rollout status deployment/fraud-serving -n mlops-demo
```

---
## 8. Entendendo os Manifests YAML

Todo recurso Kubernetes é definido em YAML. A estrutura é sempre:

```yaml
apiVersion: [versão da API do recurso]
kind: [tipo do recurso]
metadata:
  name: [nome do recurso]
  namespace: [namespace]
  labels:  # pares chave-valor para identificação e seleção
    app: minha-app
spec:
  [especificação específica do tipo de recurso]
```

### 8.1 Pod (unidade mínima)

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: meu-pod
  namespace: mlops-demo
spec:
  containers:
    - name: meu-container
      image: python:3.11-slim
      command: ["python", "-c", "print('Hello K8s')"]
```

### 8.2 Deployment (pods gerenciados)

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fraud-serving
  namespace: mlops-demo
spec:
  replicas: 2              # mantém 2 pods sempre rodando
  selector:
    matchLabels:
      app: fraud-serving   # gerencia pods com esta label
  template:                # template dos pods criados
    metadata:
      labels:
        app: fraud-serving
    spec:
      containers:
        - name: api
          image: thiagoar587/fraud-serve:latest
          ports:
            - containerPort: 8000
          env:
            - name: MLFLOW_TRACKING_URI
              value: "http://mlflow-svc:5000"
          resources:
            requests:          # mínimo garantido
              cpu: "250m"      # 250 milicores = 0.25 CPU
              memory: "512Mi"
            limits:            # máximo permitido
              cpu: "500m"
              memory: "1Gi"
          readinessProbe:      # K8s verifica: pod está pronto para tráfego?
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 10
```

### 8.3 Job (tarefa batch)

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: fraud-train-job
  namespace: mlops-demo
spec:
  backoffLimit: 0   # não tenta novamente se falhar
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: train
          image: thiagoar587/fraud-train:latest
```

### 8.4 Service (rede)

```yaml
apiVersion: v1
kind: Service
metadata:
  name: fraud-serving-svc
  namespace: mlops-demo
spec:
  selector:
    app: fraud-serving   # roteia tráfego para pods com esta label
  ports:
    - port: 8000         # porta do Service
      targetPort: 8000   # porta do container
      nodePort: 30080    # porta exposta no IP do nó (só NodePort)
  type: NodePort         # ClusterIP (interno) | NodePort (externo) | LoadBalancer
```

### 8.5 PersistentVolumeClaim (armazenamento)

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mlflow-pvc
  namespace: mlops-demo
spec:
  accessModes: ["ReadWriteOnce"]  # montado em apenas 1 nó por vez
  resources:
    requests:
      storage: 5Gi
```

---
## 9. Comandos Minikube (Desenvolvimento Local)

Minikube cria um cluster Kubernetes local para desenvolvimento e estudo.

```bash
# Verificar status do cluster
minikube status

# Iniciar o cluster
minikube start

# Iniciar com mais recursos
minikube start --cpus=4 --memory=8192

# Parar o cluster (preserva os dados)
minikube stop

# Destruir o cluster completamente
minikube delete

# Abrir o dashboard web do Kubernetes
minikube dashboard

# Ver o IP do cluster (para acessar NodePort services)
minikube ip

# Usar o Docker do minikube (para builds que o cluster veja as imagens)
eval $(minikube docker-env)

# Abrir um service no browser automaticamente
minikube service fraud-serving-svc -n mlops-demo

# Limpar espaço em disco (quando disco cheio)
docker system prune -a
minikube start
```

---
## 10. Fluxo Completo do Projeto com Todos os Comandos

### Pré-requisitos
```bash
# Verificar instalações
docker --version
kubectl version --client
minikube version

# Login no Docker Hub
docker login
```

### Etapa 1 — Preparar as Imagens
```bash
docker build -f docker/Dockerfile.mlflow -t thiagoar587/mlflow-server:latest .
docker build -f docker/Dockerfile.train  -t thiagoar587/fraud-train:latest .
docker build -f docker/Dockerfile.serve  -t thiagoar587/fraud-serve:latest .

docker push thiagoar587/mlflow-server:latest
docker push thiagoar587/fraud-train:latest
docker push thiagoar587/fraud-serve:latest
```

### Etapa 2 — Subir o Cluster
```bash
minikube start
kubectl cluster-info  # confirmar que está rodando
```

### Etapa 3 — Criar Infraestrutura
```bash
kubectl apply -f k8s/00-namespace.yaml
kubectl apply -f k8s/01-mlflow-pvc.yaml
kubectl apply -f k8s/07-data-pvc.yml
kubectl apply -f k8s/02-mlflow-deployment.yaml
kubectl apply -f k8s/03-mlflow-service.yaml

# Aguardar MLflow subir
kubectl rollout status deployment/mlflow -n mlops-demo
kubectl get pods -n mlops-demo  # STATUS deve ser Running
```

### Etapa 4 — Upload do Dataset
```bash
kubectl apply -f k8s/08-upload-pod.yaml
kubectl wait --for=condition=Ready pod/data-uploader -n mlops-demo --timeout=60s
kubectl cp data/transactions_1M.csv data-uploader:/data/transactions_1M.csv -n mlops-demo
kubectl exec data-uploader -n mlops-demo -- ls -lh /data
kubectl delete pod data-uploader -n mlops-demo
```

### Etapa 5 — Treinamento
```bash
kubectl apply -f k8s/04-train-job.yaml
kubectl logs -f job/fraud-train-job -n mlops-demo  # acompanhar em tempo real
kubectl get jobs -n mlops-demo  # COMPLETIONS deve ser 1/1
```

### Etapa 6 — Deploy da API
```bash
kubectl apply -f k8s/05-serve-deployment.yaml
kubectl apply -f k8s/06-serve-service.yaml
kubectl rollout status deployment/fraud-serving -n mlops-demo
```

### Etapa 7 — Validar
```bash
# Em terminal separado: acessa MLflow UI
kubectl port-forward svc/mlflow-svc 5000:5000 -n mlops-demo
# Abrir http://localhost:5000

# Em outro terminal: testa a API
kubectl port-forward svc/fraud-serving-svc 8000:8000 -n mlops-demo
curl http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"x": [[120.0, 14, 2, 1, 0, 0, 400, 2, 220.0, 3.2, 0, 0.15]]}'
```

---
## 11. Diagnóstico de Problemas Comuns

### Pod em CrashLoopBackOff
```bash
# 1. Ver o que aconteceu
kubectl describe pod <nome-do-pod> -n mlops-demo

# 2. Ver logs (mesmo de pod que crashou)
kubectl logs <nome-do-pod> -n mlops-demo

# 3. Ver logs da tentativa anterior
kubectl logs <nome-do-pod> -n mlops-demo --previous
```

### Pod em Pending (não consegue ser alocado)
```bash
# Describe mostra o motivo (falta de recursos, PVC não montado, etc.)
kubectl describe pod <nome-do-pod> -n mlops-demo
# Checar seção "Events" no final do output
```

### Serviço não acessível
```bash
# Verificar se o selector bate com os labels dos pods
kubectl describe svc fraud-serving-svc -n mlops-demo
kubectl get pods -n mlops-demo --show-labels

# Verificar endpoints (deve ter IPs dos pods)
kubectl get endpoints -n mlops-demo
```

### Disco cheio (minikube)
```bash
docker system prune -a   # libera espaço
minikube start           # reinicia
```

### PVC não monta (Pending)
```bash
kubectl get pvc -n mlops-demo
kubectl describe pvc mlflow-pvc -n mlops-demo
# Status Pending = StorageClass não conseguiu provisionar o disco
# No minikube, isso costuma se resolver sozinho ao iniciar o cluster
```

---
## 12. Resumo: Docker vs Kubernetes

| | Docker | Kubernetes |
|---|---|---|
| **O que é** | Plataforma de containers | Orquestrador de containers |
| **Escala** | Uma máquina | Cluster de máquinas |
| **CLI** | `docker` | `kubectl` |
| **Unidade básica** | Container | Pod |
| **Declarativo?** | Dockerfile (build) | YAML manifests (estado desejado) |
| **Auto-recuperação** | Não (manual) | Sim (automática) |
| **Escalonamento** | Manual | Automático (HPA) |
| **Service discovery** | Redes + links | DNS interno |
| **Persistência** | Volumes locais | PersistentVolumes |

### Mapa Mental dos Comandos

```
DOCKER                          KUBERNETES (kubectl)
──────────────────────          ──────────────────────────────────
docker build    → imagem        kubectl apply -f   → cria/atualiza recurso
docker push     → registry      kubectl delete -f  → remove recurso
docker pull     → local         kubectl get        → lista recursos
docker run      → container     kubectl describe   → detalhes + eventos
docker ps       → lista         kubectl logs       → logs dos pods
docker logs     → logs          kubectl exec       → shell no pod
docker exec     → shell         kubectl cp         → copia arquivos
docker stop/rm  → remove        kubectl port-forward → acesso local
docker images   → imagens       kubectl scale      → escala réplicas
docker system prune → limpa     kubectl rollout    → manage deploys
```